#### Simple Tokenization (V1 and V2) of the verdict short story

In [50]:
file_path = "the-verdict.txt" # phase-1-foundations/tokenization-chap2/the-verdict.txt
with open(file_path, "r") as file:
    raw_text = file.read()
print(f"Total number of characters: {len(raw_text)}")
print(raw_text[:99])  # first 100 characters 

Total number of characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [51]:
import re

text = "Hello world. This, and this, is a test."
result = re.split(r'(\s)', text)
# remove whitespace for now, include in later tokeization techniques
result = [item for item in result if item.strip() != ''] 
print(result)

['Hello', 'world.', 'This,', 'and', 'this,', 'is', 'a', 'test.']


In [52]:
import re

# handle more punctuation.
text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [53]:
# Now, let us apply it to the short story
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed)) # first 30 tokenized words.

4690


In [54]:
print(preprocessed[:30]) # first 30 tokenized words.

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [55]:
# Let us find the unique words, which is vocabulary
vocab = sorted(set(preprocessed))
vocab_size = len(vocab)
print(f"Vocabulary size: {vocab_size}")

Vocabulary size: 1130


In [56]:
vocab = {token:integer for integer,token in enumerate(vocab)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


In [57]:
# Create an inverse version of the tokenizer, a dictionary that maps integers to tokens 
# based on the vocab dict above.
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()} 

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    
    def decode(self, ids):        
            text = " ".join([self.int_to_str[i] for i in ids]) 
            
            text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)   
            return text

In [58]:
# Test the simple tknizer class
tokenizer = SimpleTokenizerV1(vocab)
test_text =  """"It's the last he painted, you know," 
       Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(test_text)
print(f"Encoded ids: {ids}")

Encoded ids: [1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [59]:
# Reverse:
print(f"Decoded text from ids: {tokenizer.decode(ids)}")

Decoded text from ids: " It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [60]:
# Add special context tokens:
vocab = sorted(set(preprocessed))
vocab.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:integer for integer,token in enumerate(vocab)}
print(len(vocab.items()))

1132


In [61]:
# Updating the tknizer to include spl and unknown tkns.
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        preprocessed = [item if item in self.str_to_int           
                        else "<|unk|>" for item in preprocessed]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)   
        return text

In [62]:
# test
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = f"{text1} <|endoftext|> {text2}"

tokenizer2 = SimpleTokenizerV2(vocab)
ids = tokenizer2.encode(text)
print(f"Encoded ids: {ids}")

Encoded ids: [1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


In [63]:
# reverse:
print(tokenizer2.decode(ids))

<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


#### Byte Pair Encoding Tokenizer

In [64]:
# ! uv pip install tiktoken

In [65]:
from importlib.metadata import version

import tiktoken

print(f"tiktoken version: {version('tiktoken')}")


tiktoken version: 0.14.0


In [66]:
# instantiate
tokenizer = tiktoken.get_encoding("gpt2")

In [67]:
# test
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     " of someunknownPlace."
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]


In [68]:
strings = tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


In [69]:
# Exercise 2.1
text_exercise = "Akwirw ier"
integers_exercise = tokenizer.encode(text_exercise, allowed_special={"<|endoftext|>"})
strings_exercise = tokenizer.decode(integers_exercise)
print(f"encoded: {integers_exercise}\ndecoded: {strings_exercise}")

encoded: [33901, 86, 343, 86, 220, 959]
decoded: Akwirw ier


#### Data Sampling with a Sliding Window 

In [70]:
# Creating input output pairs from the verdict
file_path = "the-verdict.txt" # phase-1-foundations/tokenization-chap2/the-verdict.txt
with open(file_path, "r", encoding="utf-8") as file:
    raw_text = file.read()
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [71]:
enc_sample = enc_text[50:]

In [72]:
# create sample pairs
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x: {x}\ny:       {y}")

x: [290, 4920, 2241, 287]
y:       [4920, 2241, 287, 257]


In [73]:
# Print the desired ip pairs
print("Context ---> Desired (token ID's)")
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(f"{context} ---> {desired}")

Context ---> Desired (token ID's)
[290] ---> 4920
[290, 4920] ---> 2241
[290, 4920, 2241] ---> 287
[290, 4920, 2241, 287] ---> 257


In [74]:
# Print the desired ip pairs
print("Context ---> Desired (actual text)")
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(f"{tokenizer.decode(context)} ---> {tokenizer.decode([desired])}")

Context ---> Desired (actual text)
 and --->  established
 and established --->  himself
 and established himself --->  in
 and established himself in --->  a


In [75]:
# ! uv pip install torch

In [76]:
# Structure the custom dataloader from Dataloader class.
import torch
from torch.utils.data import Dataset


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids) - max_length, stride):    
            input_chunk = token_ids[i:i + max_length] # sliding window
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):   
        return len(self.input_ids) # return the total rows of dataset
    
    def __getitem__(self, idx):        
        return self.input_ids[idx], self.target_ids[idx] # return single row of dataset

In [77]:
# implementation
from torch.utils.data import DataLoader


def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True,drop_last=True, 
                         num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride) # create dataset from our defn.
    dataloader = DataLoader(
        dataset, 
        batch_size= batch_size,
        shuffle=shuffle,
        drop_last=drop_last,    # drops the last batch when shorter than batch_size
        num_workers=num_workers, # number of processes
    )
    return dataloader

In [78]:
# test now
with open("the-verdict.txt", "r", encoding="utf-8") as file:
    raw_text = file.read()

dataloader = create_dataloader_v1(
    raw_text, batch_size=1,max_length=4, stride=1, shuffle=False
    )
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [79]:
# Exercise 2.2
# max_length = 2; stride = 2
data_loader_exercise1 = create_dataloader_v1(
    raw_text, batch_size=1, max_length=2, stride=2, shuffle=False
)
data_iter1 = iter(data_loader_exercise1)
first_batch1 = next(data_iter1)
print(first_batch1)

# max_length = 8; stride = 2
data_loader_exercise2 = create_dataloader_v1(
    raw_text, batch_size=1, max_length=8, stride=2, shuffle=False
)
data_iter2 = iter(data_loader_exercise2)
first_batch2 = next(data_iter2)
print(first_batch2)

[tensor([[ 40, 367]]), tensor([[ 367, 2885]])]
[tensor([[  40,  367, 2885, 1464, 1807, 3619,  402,  271]]), tensor([[  367,  2885,  1464,  1807,  3619,   402,   271, 10899]])]


In [80]:
# Batch size experimentation:
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=4, stride=4,
    shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)


Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


#### Embedding creation

In [81]:
# example dummy embedding creation tiny scale
from torch.nn import Embedding

input_ids = torch.tensor([2, 3, 5, 1])

# Vocab size: 6, random_seed: 123 (for reproducibility)
vocab_size_test = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = Embedding(vocab_size_test, output_dim)
print(embedding_layer.weight) # prints the weight matrix of the embedding layer.

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [82]:
# apply to a token id to the embedding and then obtain a vector
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


In [83]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


#### Implementing Positional Embedding (Encoding Word Positions):

In [84]:
# vocab size and output dim embeddings
vocab_size = 50257
output_dim = 256
token_embedding_layer = Embedding(vocab_size, output_dim)

# creating the data loader.
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
   stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [87]:
# Check the dimension of the embedding: 8 x 4 x 256
# batch size is 8 and the num tokens 4 and 256 dimensional vector (output_dim)
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [90]:
# Absolute positional embedding approach
context_length = max_length
pos_embedding_layer = Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [ ]:
# append positional embedding vec to token embedding:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])


In [100]:
# sample embeddings preview from the whole matrix:
sample_embedding = input_embeddings[0]
print(sample_embedding) # prints 4 vectors of length 256 each.
print(f"\n Length of the embedding: {sample_embedding.shape}")

tensor([[-0.3281,  1.6782,  0.6298,  ..., -0.2670, -1.6620,  0.2165],
        [ 0.8674, -0.6925, -0.6063,  ...,  1.2927,  0.5018, -0.6181],
        [-0.4869, -1.7733, -0.3802,  ...,  2.0836,  2.6533,  0.9384],
        [-0.0091,  1.3356,  1.8771,  ..., -1.8233,  0.9045,  1.6972]],
       grad_fn=<SelectBackward0>)

 Length of the embedding: torch.Size([4, 256])
